# PA3 — weights + characteristics, 0CQ single frequency

Holdings-based snapshot from PA Engine, as of the most recent calendar quarter end.
Companion to `spar_composite_returns_template.ipynb`, which is returns-based (SPAR) and
needs no holdings.

**Still needed to run:** `PA_DOCUMENT` and the two component names. Everything else —
account paths, holdings modes, benchmark ids — is read off the saved components by Cell 4c
and printed as a paste-ready config block.

## Call structure — one unit per distinct port + bench pair

| Unit | Account | Benchmark |
|---|---|---|
| `<tile>__LC` | LC | Russell 1000 |
| `<tile>__LCS` | LCS | Russell 1000 |
| `<tile>__SMID` | SMID | Russell 2500 |
| `<tile>__CONC` | CONC | Russell 3000 |

Two tiles x four pairs = **8 units in one multi-port POST**.

One account and one benchmark per unit buys two things. The account↔benchmark pairing is
unambiguous — worth having, because `PACalculationParameters` takes both `accounts` and
`benchmarks` as *lists*, and with 4 accounts against 3 benchmarks in one unit the counts
don't match, so positional pairing is impossible and the schema doesn't say whether PA
cross-products them. And the **strategy is recoverable from the unit key**, so no output
column has to be reverse-engineered to find out which portfolio a row belongs to.

## Weights: `GROUPSALL`, and what comes out of it

`GROUPSALL` returns group rows **and** security rows in one response, so a single call
yields both sector-level weights (portfolio / benchmark / active) and security-level
portfolio weights for top-N sorting.

Target columns on each holding row:

| Column | Source |
|---|---|
| FSYM perm id | component column (default) |
| **GICS sector** | **derived from the STACH grouping** — see below |
| cash flag | component column, or derived |
| ultimate parent FSYM id | component column, if exposed |

### GICS sector is a grouping, not a column

In `GROUPSALL` output the sector isn't an attribute of a security row — the security rows
are *nested under* their sector's group row. So sector has to be projected down onto each
holding to sit in an adjacent column. Cell 7 does it two ways, in order:

1. If a grouping column is already populated on security rows, use it directly.
2. Otherwise forward-fill the sector label from each preceding group row.

> ⚠️ **Path 2 depends on STACH emitting rows in hierarchical order** — every security
> immediately following its parent group. That holds for PA's grouped output but it is an
> assumption about row order, not a documented guarantee. Cell 7 reports which path was
> used and how many holdings got a sector; check the sector distribution against the
> workstation for one composite before trusting it. A silent mis-fill here mislabels every
> holding's sector without erroring.

### Mixed grain is a double-counting hazard

A sector row's weight is the sum of its children's, so any unfiltered `SUM` over the
combined output roughly doubles every total. Cell 8 therefore splits by grain into
`pa_sector_weights` and `pa_security_weights` rather than landing one ambiguous table.

`componentdetail` accepts `GROUPS`, `GROUPSALL`, `TOTALS`, or `SECURITIES`.

## Fee basis does not apply

Weights and characteristics are holdings attributes — no gross/net distinction. The 2x
fee-basis fan-out from the SPAR notebook is absent by design.

## Versions

| Package | Version |
|---|---|
| `fds.sdk.PAEngine` | **4.0.0** (upstream latest, 2026-07-21) |
| `fds.sdk.utils` | 3.0.1 |
| `fds.protobuf.stach.extensions` | 1.3.3 |
| `deltalake` | 1.6.2 |

**PAEngine 4.0.0 carries a breaking change beyond the 2026-05-20 Python-wide bump.** Per
upstream `BREAKING.md` (2026-07-21), `PADateParameters.enddate` and `.frequency` lost
`required: true` and the schema's `required` array was dropped, which **reshuffles
positional arguments** across ~6 operations including the calculation endpoints and
`convertPADatesToAbsoluteFormat`. Every call here passes keywords, so it is unaffected —
but older PA code passing dates positionally will silently bind the wrong values.

> ⚠️ The PA SDK vendored under `code/python/PAEngine/v3/` in this repo is **2.2.2** and
> predates all of the above. Verify against upstream `main`.

Attach libraries to a **Fabric Environment**, not `%pip` — inline installs are disabled by
default in pipeline runs and unsupported in reference runs. Interactive first run only:
```
%pip install fds.sdk.PAEngine==4.0.0 fds.sdk.utils==3.0.1 \
             fds.protobuf.stach.extensions==1.3.3 deltalake==1.6.2
```

In [ ]:
# === Cell 1: credentials ===================================================
%run HBCM_Config

In [ ]:
# === Cell 2: imports + API client ==========================================
import json, time, datetime as dt
import pandas as pd

import fds.sdk.PAEngine
from fds.sdk.PAEngine.api import (
    pa_calculations_api, components_api, accounts_api,
    columns_api, groups_api, frequencies_api, dates_api,
)
from fds.sdk.PAEngine.models import (
    PACalculationParametersRoot, PACalculationParameters,
    PAIdentifier, PADateParameters, CalculationMeta,
)
from urllib3 import Retry

SDK_VERSION = fds.sdk.PAEngine.__version__
assert int(SDK_VERSION.split(".")[0]) >= 4, (
    f"fds.sdk.PAEngine {SDK_VERSION} found; this notebook targets >=4.0.0 "
    "(PADateParameters changed shape). Check the bound Fabric Environment."
)
print("PAEngine SDK", SDK_VERSION)

configuration = fds.sdk.PAEngine.Configuration(
    username=FACTSET_USER, password=FACTSET_APIKEY,
)
configuration.retries = Retry(
    total=3, status_forcelist=[500, 502, 503, 504], backoff_factor=2,
    allowed_methods=frozenset(["GET", "POST"]),
)

api_client = fds.sdk.PAEngine.ApiClient(configuration)
calc_api = pa_calculations_api.PACalculationsApi(api_client)
comp_api = components_api.ComponentsApi(api_client)

In [ ]:
# === Cell 3: THE CONFIG BLOCK ==============================================

CURRENCY = "USD"

# Point-in-time snapshot: one date, no series. Single frequency means startdate does not
# shape the result, but PA wants a coherent window, so it is set to the same date.
AS_OF_RELATIVE = "0CQ"
FREQUENCY = "Single"
USE_ABSOLUTE_AS_OF = False

def _prior_quarter_end(today=None):
    d = today or dt.date.today()
    qe = dt.date(d.year, ((d.month - 1) // 3) * 3 + 1, 1) - dt.timedelta(days=1)
    return qe.strftime("%Y%m%d")

AS_OF_ABS = _prior_quarter_end()
AS_OF = AS_OF_ABS if USE_ABSOLUTE_AS_OF else AS_OF_RELATIVE

# --- the document (the one thing that must be right) -----------------------
PA_DOCUMENT = "<TODO PA3 document path>"

TILES = {
    "weights": {
        "component_name": "<TODO exact workstation name>", "pinned_componentid": None,
        # GROUPSALL -> group rows AND security rows in one response.
        "componentdetail": "GROUPSALL",
        "split_grain": True,
    },
    "characteristics": {
        "component_name": "<TODO exact workstation name>", "pinned_componentid": None,
        "componentdetail": "GROUPS",
        "split_grain": False,
    },
}

# --- benchmarks ------------------------------------------------------------
# Defined once so LC and LCS cannot drift onto different benchmarks. Units are still built
# per port+bench PAIR, not per group — see Cell 5.
BENCHMARK_GROUPS = {
    "r1000": {"label": "Russell 1000", "id": "<TODO>"},   # LC + LCS
    "r2500": {"label": "Russell 2500", "id": "<TODO>"},   # SMID
    "r3000": {"label": "Russell 3000", "id": "<TODO>"},   # CONC
}

# --- accounts --------------------------------------------------------------
# PA needs the HOLDINGS account path, not the returns ACCT the SPAR notebook uses — they
# are different objects, so do not copy one across.
# holdingsmode: B&H, TBR, OMS, EXT or VLT.
# Cell 4c reads these off the saved component and prints them ready to paste.
STRATEGIES = {
    "LC":   {"label": "Large Cap",           "acct": "<TODO>",
             "holdingsmode": "B&H", "bench_group": "r1000"},
    "SMID": {"label": "SMID",                "acct": "<TODO>",
             "holdingsmode": "B&H", "bench_group": "r2500"},
    "LCS":  {"label": "Large Cap Select",    "acct": "<TODO>",
             "holdingsmode": "B&H", "bench_group": "r1000"},
    "CONC": {"label": "Concentrated Equity", "acct": "<TODO>",
             "holdingsmode": "B&H", "bench_group": "r3000"},
}

# --- output column mapping -------------------------------------------------
# STACH column labels come from the component, so they cannot be known in advance. Each
# entry is a list of candidate names tried in order; the first match wins and is renamed to
# the canonical key. Add the real name to the front once Cell 7 prints the actual columns.
COLUMN_HINTS = {
    "fsym_perm_id": [
        "fsym_perm_id", "fsym_id", "fsym_security_id", "fsym_regional_id", "perm_id",
    ],
    "gics_sector": [          # only if the component exposes it directly as a column;
        "gics_sector", "sector",   # otherwise Cell 7 derives it from the grouping
    ],
    "cash_flag": [
        "cash_flag", "is_cash", "cash", "asset_class", "security_type",
    ],
    "ultimate_parent_fsym_id": [   # optional
        "ultimate_parent_fsym_id", "fsym_ultimate_parent_id", "ult_parent_fsym_id",
        "ultimate_parent_id",
    ],
}
REQUIRED_CANONICAL = ["fsym_perm_id"]      # ultimate parent is "possibly", so not here

# --- OneLake target --------------------------------------------------------
WORKSPACE_ID = "1b9fac18-9d75-4437-ab6c-b6ba44ff46a8"   # HBCM - Production
LAKEHOUSE_ID = "7cdf13b1-4586-4a02-b8ff-72fcf6db1277"   # hbcm_datahub
ONELAKE = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}"
RAW_DIR = f"{ONELAKE}/Files/raw/pa"
# Separate tables per grain — a sector row's weight is the sum of its securities'.
TABLE_SECTOR = f"{ONELAKE}/Tables/factset/pa_sector_weights"
TABLE_SECURITY = f"{ONELAKE}/Tables/factset/pa_security_weights"
TABLE_CHARACTERISTICS = f"{ONELAKE}/Tables/factset/pa_characteristics"

def benchmark_of(code: str) -> dict:
    return BENCHMARK_GROUPS[STRATEGIES[code]["bench_group"]]

assert all(s["bench_group"] in BENCHMARK_GROUPS for s in STRATEGIES.values())
print(f"as-of sent: {AS_OF!r}  frequency: {FREQUENCY}  asof_date written: {AS_OF_ABS}")
print(f"{len(TILES)} tiles x {len(STRATEGIES)} port+bench pairs = "
      f"{len(TILES) * len(STRATEGIES)} units")
for c, s in STRATEGIES.items():
    print(f"  {c:<6} {s['acct']:<28} vs {benchmark_of(c)['label']}")

In [ ]:
# === Cell 4: resolve dates + component ids, every run =====================

def _field(obj, name):
    """SDK models allow attribute or dict-style access depending on construction."""
    if hasattr(obj, name):
        return getattr(obj, name)
    try:
        return obj.get(name)
    except AttributeError:
        return None

# (a) Resolve 0CQ server-side and compare against the locally computed quarter end.
#     PA has a DatesApi; SPAR does not. This is the check the SPAR notebook cannot run.
d_api = dates_api.DatesApi(api_client)
try:
    print("PA resolved dates:",
          d_api.convert_pa_dates_to_absolute_format(startdate=AS_OF, enddate=AS_OF))
    print(f"locally computed quarter end: {AS_OF_ABS}")
except Exception as e:
    # 4.0.0 reshuffled this signature and added an optional `calendar` parameter. If this
    # errors, check the current arg list before concluding 0CQ is wrong.
    print(f"date conversion unavailable ({e!r}); relying on AS_OF_ABS for labelling")

# (b) component name -> live id. Ids are not stable: re-saving a component can mint a new
#     one, and a stale id 400s with nothing pointing at the id as the cause.
def resolve_component_ids(document=PA_DOCUMENT):
    summary = comp_api.get_pa_components(document=document)
    by_name = {}
    for cid, meta in (summary.data or {}).items():
        by_name.setdefault(_field(meta, "name"), []).append(cid)

    resolved, drift, missing = {}, [], []
    for tile, cfg in TILES.items():
        hits = by_name.get(cfg["component_name"], [])
        if len(hits) != 1:
            missing.append((tile, cfg["component_name"], len(hits)))
            continue
        resolved[tile] = hits[0]
        if cfg.get("pinned_componentid") and cfg["pinned_componentid"] != hits[0]:
            drift.append((tile, cfg["component_name"], cfg["pinned_componentid"], hits[0]))
    return resolved, drift, missing, by_name

RESOLVED_COMPONENTS, COMPONENT_DRIFT, COMPONENT_MISSING, COMPONENTS_BY_NAME = \
    resolve_component_ids()

for tile, cid in RESOLVED_COMPONENTS.items():
    print(f"{tile:<18} {cid}  ({TILES[tile]['component_name']})")

if COMPONENT_DRIFT:
    print("\n*** COMPONENT ID DRIFT — the component was re-saved. Confirm its columns")
    print("*** still match, then update pinned_componentid:")
    for tile, name, was, now in COMPONENT_DRIFT:
        print(f"    {tile}: {name!r}  {was} -> {now}")

if COMPONENT_MISSING:
    print("\nUnresolved tiles:", COMPONENT_MISSING)
    print("Available component names in", PA_DOCUMENT)
    for name, cids in sorted(COMPONENTS_BY_NAME.items(), key=lambda kv: str(kv[0])):
        print(f"    {name!r}: {cids}")

assert not COMPONENT_MISSING, "every tile needs exactly one matching component name"

In [ ]:
# === Cell 4c: read the saved config off each component ====================
# PAComponent exposes the accounts, benchmarks, currency, dates and snapshot flag SAVED IN
# THE DOCUMENT. So the document path plus two component names is enough to fill in the
# account paths, holdings modes and benchmark ids — no need to hand-transcribe them from
# the workstation. Run once, paste the result into Cell 3, then skip.

for tile, cid in RESOLVED_COMPONENTS.items():
    try:
        comp = comp_api.get_pa_component_by_id(id=cid)
    except fds.sdk.PAEngine.ApiException as e:
        print(f"{tile}: lookup failed {e.status} {e.body}")
        continue
    d = comp.data
    print(f"\n=== {tile} ({_field(d, 'name')}) ===")
    print(f"  path      {_field(d, 'path')}")
    print(f"  currency  {_field(d, 'currencyisocode')}")
    # snapshot=True means the component is a point-in-time snapshot rather than a
    # subperiod calculation — which is exactly what a 0CQ weights pull wants. If this is
    # False for the weights component, "Single" frequency may not mean what you expect.
    print(f"  snapshot  {_field(d, 'snapshot')}")
    print(f"  dates     {_field(d, 'dates')}")
    for a in (_field(d, "accounts") or []):
        print(f"  ACCOUNT   id={_field(a, 'id')!r} holdingsmode={_field(a, 'holdingsmode')!r}")
    for b in (_field(d, "benchmarks") or []):
        print(f"  BENCHMARK id={_field(b, 'id')!r} holdingsmode={_field(b, 'holdingsmode')!r}")

print("\nMap each ACCOUNT id to a strategy code in Cell 3's STRATEGIES, and each")
print("BENCHMARK id to the matching entry in BENCHMARK_GROUPS.")

In [ ]:
# === Cell 4d: DISCOVERY — optional lookups ================================
a_api = accounts_api.AccountsApi(api_client)
col_api = columns_api.ColumnsApi(api_client)
grp_api = groups_api.GroupsApi(api_client)
frq_api = frequencies_api.FrequenciesApi(api_client)

# Weight and identifier columns. GROUPSALL sets the ROW GRAIN, not which columns exist —
# if active weight, FSYM perm id, cash flag or ultimate parent are missing from the
# output, the component doesn't expose them. Either edit the component or override
# PACalculationParameters.columns on the request.
#   print(col_api.get_pa_columns(name="weight", category="", directory=""))
#   print(col_api.get_pa_columns(name="fsym", category="", directory=""))

# Groupings — what the sector rows are actually grouped by. PA uses the component's saved
# grouping unless PACalculationParameters.groups overrides it. Confirm it is GICS sector
# and not some other scheme, because Cell 7 projects it onto every holding.
#   print(grp_api.get_pa_groups())

#   print(frq_api.get_pa_frequencies())        # confirm "Single"
#   print(a_api.get_accounts(path="Client:/")) # browse holdings accounts
print("uncomment the lookup you need")

In [ ]:
# === Cell 5: build units — one per distinct port + bench pair =============
# 2 tiles x 4 pairs = 8 units, submitted as one multi-port POST.
#
# One account + one benchmark per unit means the pairing is explicit (no inference about
# how PA matches a list of accounts to a list of benchmarks) and the strategy is carried by
# the unit key, so no output column has to be reverse-engineered to identify the portfolio.

def pa_dates() -> PADateParameters:
    # Keyword args throughout: 4.0.0 dropped `required` on enddate/frequency, which
    # reshuffled positional arguments across the PA calculation endpoints.
    return PADateParameters(startdate=AS_OF, enddate=AS_OF, frequency=FREQUENCY)

def build_unit(tile_name: str, tile_cfg: dict, code: str) -> PACalculationParameters:
    s = STRATEGIES[code]
    return PACalculationParameters(
        componentid=RESOLVED_COMPONENTS[tile_name],
        accounts=[PAIdentifier(id=s["acct"], holdingsmode=s["holdingsmode"])],
        benchmarks=[PAIdentifier(id=benchmark_of(code)["id"])],
        dates=pa_dates(),
        currencyisocode=CURRENCY,
        componentdetail=tile_cfg["componentdetail"],
    )

UNIT_KEYS, units = {}, {}
for tile_name, tile_cfg in TILES.items():
    for code in STRATEGIES:
        key = f"{tile_name}__{code}"
        units[key] = build_unit(tile_name, tile_cfg, code)
        UNIT_KEYS[key] = (tile_name, code)

params_root = PACalculationParametersRoot(
    data=units,
    meta=CalculationMeta(
        contentorganization="SimplifiedRow",
        stach_content_organization="SimplifiedRow",
        contenttype="Json",
        format="JsonStach",
    ),
)
for key, (tile_name, code) in UNIT_KEYS.items():
    print(f"{key:<26} {TILES[tile_name]['componentdetail']:<10} "
          f"{STRATEGIES[code]['acct']:<26} vs {benchmark_of(code)['label']}")

In [ ]:
# === Cell 6: submit + poll ================================================
# 200 sync, 201 ready, 202 poll. Multi-unit always returns 202.

def run_pa(params_root, deadline=10, poll_interval=3, timeout=900):
    wrapper = calc_api.post_and_calculate(
        x_fact_set_api_long_running_deadline=deadline,
        pa_calculation_parameters_root=params_root,
    )
    code = wrapper.get_status_code()
    if code == 200:
        status_root = wrapper.get_response_200()
    elif code == 201:
        status_root = wrapper.get_response_201()
    elif code == 202:
        status_root = wrapper.get_response_202()
        calc_id = status_root.data.calculationid
        deadline_at = time.time() + timeout
        while True:
            if time.time() > deadline_at:
                calc_api.cancel_calculation_by_id(id=calc_id)
                raise TimeoutError(f"calc {calc_id} exceeded {timeout}s (cancelled)")
            poll = calc_api.get_calculation_status_by_id(id=calc_id)
            if poll.get_status_code() == 200:
                status_root = poll.get_response_200()
                break
            if poll.get_status_code() != 202:
                raise RuntimeError(f"unexpected poll status {poll.get_status_code()}")
            time.sleep(poll_interval)
    else:
        raise RuntimeError(f"unexpected submit status {code}")

    calc_id = status_root.data.calculationid
    out = []
    for unit_id, unit_status in (status_root.data.units or {}).items():
        st = getattr(unit_status, "status", None)
        if st != "Success":
            out.append((unit_id, None, st))
            continue
        out.append((unit_id,
                    calc_api.get_calculation_unit_result_by_id(id=calc_id, unit_id=unit_id),
                    st))
    return calc_id, out

calc_id, results = run_pa(params_root)
failed = [(u, s) for u, r, s in results if r is None]
print(f"calc={calc_id} ok={len(results) - len(failed)} failed={len(failed)}")
for u, s in failed:
    print(f"  FAILED {u}: {s}")
assert results and not failed, "resolve failures before writing to the lakehouse"

In [ ]:
# === Cell 7: raw landing, STACH parse, sector projection ==================
from fds.protobuf.stach.extensions.StachExtensionFactory import StachExtensionFactory
from fds.protobuf.stach.extensions.StachVersion import StachVersion

asof_tag = AS_OF_ABS
for unit_id, res, _ in results:
    notebookutils.fs.put(f"{RAW_DIR}/asof={asof_tag}/{unit_id}.json",
                         json.dumps(res.to_dict(), default=str), True)
print(f"landed {len(results)} raw payloads")

def stach_to_dataframes(api_response):
    ext = StachExtensionFactory.get_stach_extension(StachVersion.V2)
    return [pd.DataFrame(t.data, columns=t.columns)
            for t in ext.convert(json.dumps(api_response.to_dict(), default=str))]

def norm(cols):
    return [str(c).strip().replace(" ", "_").replace("-", "_").lower() for c in cols]

GROUP_LEVEL_COLS = ("level", "depth", "hierarchy_level", "row_level")
SECURITY_ID_COLS = ("fsym_perm_id", "fsym_id", "security", "security_id", "asset_id",
                    "symbol", "ticker")
GROUPING_COLS = ("gics_sector", "sector", "group", "grouping", "group_name", "group_1")

def first_present(df, candidates):
    return next((c for c in candidates if c in df.columns), None)

def group_mask(df):
    """True on sector/group rows, False on security rows. None if undecidable.

    HEURISTIC — confirm against real output. Getting it wrong silently double-counts,
    because a sector row's weight is the sum of its children's.
    """
    lvl = first_present(df, GROUP_LEVEL_COLS)
    if lvl:
        return pd.to_numeric(df[lvl], errors="coerce").fillna(-1) == 0
    sid = first_present(df, SECURITY_ID_COLS)
    if sid:
        s = df[sid].astype("string").str.strip()
        return s.isna() | (s == "")
    return None

def project_sector(df):
    """Put the GICS sector on every holding row, in an adjacent column.

    In GROUPSALL output the sector is a GROUPING, not an attribute of the security — the
    security rows sit under their sector's group row. Two paths, in order:
      1. a grouping column already populated on security rows -> use it
      2. forward-fill the label down from each preceding group row

    Path 2 assumes STACH emits rows in hierarchical order, each security immediately after
    its parent group. True of PA's grouped output, but an assumption about row order rather
    than a documented guarantee — hence the reporting below.
    """
    gcol = first_present(df, GROUPING_COLS)
    mask = group_mask(df)
    if gcol is None or mask is None:
        df["gics_sector"] = pd.NA
        return df, "none"

    labels = df[gcol].astype("string").str.strip()
    populated_on_securities = labels[~mask].notna() & (labels[~mask] != "")
    if len(labels[~mask]) and populated_on_securities.all():
        df["gics_sector"] = labels
        return df, f"direct:{gcol}"

    # Forward-fill: keep the label only where it is a group row, then ffill downward.
    df["gics_sector"] = labels.where(mask).ffill()
    return df, f"ffill:{gcol}"

frames, sector_methods = [], {}
for unit_id, res, _ in results:
    tile_name, code = UNIT_KEYS[unit_id]
    bmk = benchmark_of(code)
    for i, df in enumerate(stach_to_dataframes(res)):
        df = df.copy()
        df.columns = norm(df.columns)

        # Canonicalise the identifier columns before anything reads them.
        for canonical, candidates in COLUMN_HINTS.items():
            hit = first_present(df, [c for c in candidates if c != canonical])
            if canonical not in df.columns and hit:
                df = df.rename(columns={hit: canonical})

        if TILES[tile_name]["split_grain"]:
            df, method = project_sector(df)
            sector_methods[f"{unit_id}[{i}]"] = method

        for pos, (col, val) in enumerate([
            ("asof_date",       asof_tag),
            ("tile",            tile_name),
            ("strategy_code",   code),                  # from the unit key, not the output
            ("strategy",        STRATEGIES[code]["label"]),
            ("account_id",      STRATEGIES[code]["acct"]),
            ("benchmark_id",    bmk["id"]),
            ("benchmark_label", bmk["label"]),
            ("componentid",     RESOLVED_COMPONENTS[tile_name]),
            ("componentdetail", TILES[tile_name]["componentdetail"]),
            ("table_ix",        i),
        ]):
            df.insert(pos, col, val)
        frames.append(df)

tidy = pd.concat(frames, ignore_index=True)

print(f"\n{tidy.shape[0]} rows, {tidy.shape[1]} columns")
print("\ncolumns:", list(tidy.columns))
print("\nsector projection method per unit/table:")
for k, v in sector_methods.items():
    print(f"    {k:<34} {v}")

# Which of the requested columns actually arrived.
print("\nrequested columns present:")
for canonical in list(COLUMN_HINTS) + ["gics_sector"]:
    present = canonical in tidy.columns
    filled = int(tidy[canonical].notna().sum()) if present else 0
    flag = "" if present else "   <-- MISSING, add the real name to COLUMN_HINTS"
    print(f"    {canonical:<26} present={present} non_null={filled}{flag}")

_w = tidy[tidy["tile"] == "weights"]
_m = group_mask(_w) if len(_w) else None
print(f"\nweights grain: discriminator_found={_m is not None}"
      + (f" group_rows={int(_m.sum())} security_rows={int((~_m).sum())}" if _m is not None else ""))
if _m is not None and "gics_sector" in _w.columns:
    print("\nholdings per sector (sanity-check against the workstation):")
    print(_w[~_m].groupby(["strategy_code", "gics_sector"], dropna=False).size())
display(tidy.head(30))

In [ ]:
# === Cell 8: split by grain and write =====================================
# Guarded rather than commented out: the asserts below fail loudly on first run until the
# grain discriminator and column names are confirmed, which is the behaviour you want —
# a mixed-grain table double-counts silently and a sector-less holdings table looks fine.
from deltalake import DeltaTable, write_deltalake

weights = tidy[tidy["tile"] == "weights"].copy()
chars = tidy[tidy["tile"] == "characteristics"].copy()

mask = group_mask(weights)
assert mask is not None, (
    "no grain discriminator found in the GROUPSALL output — inspect Cell 7's column list "
    "and extend GROUP_LEVEL_COLS / SECURITY_ID_COLS"
)
for canonical in REQUIRED_CANONICAL:
    assert canonical in weights.columns, (
        f"{canonical} absent — add its real STACH name to COLUMN_HINTS, or the component "
        f"does not expose it"
    )
securities = weights[~mask]
assert securities["gics_sector"].notna().all(), (
    "some holdings have no GICS sector — the grouping projection in Cell 7 did not cover "
    "every security row; check the reported method before writing"
)

def write(path, df, name):
    if df.empty:
        print(f"skip {name}: no rows")
        return
    df = df.astype({c: "string" for c in df.select_dtypes("object").columns})
    try:
        DeltaTable(path).delete(f"asof_date = '{asof_tag}'")
        mode = "append"
    except Exception:
        mode = "overwrite"          # table does not exist yet
    write_deltalake(path, df, mode=mode, schema_mode="merge")
    print(f"wrote {len(df)} rows to {name} (mode={mode})")

write(TABLE_SECTOR, weights[mask], "factset.pa_sector_weights")
write(TABLE_SECURITY, securities, "factset.pa_security_weights")
write(TABLE_CHARACTERISTICS, chars, "factset.pa_characteristics")

# Re-frame any Direct Lake semantic model BEFORE running VACUUM — vacuuming files a framed
# model still points at gives query errors on missing files. Order: write -> frame -> vacuum.

## To finish

1. `PA_DOCUMENT` + the two `component_name`s in Cell 3.
2. Run Cell 4c — it prints the accounts, holdings modes and benchmarks saved in each
   component. Map them into `STRATEGIES` and `BENCHMARK_GROUPS`.
3. Run to Cell 7 and read its three reports: which requested columns arrived, which
   sector-projection path was used, and the holdings-per-sector counts.
4. Add any missing real column names to `COLUMN_HINTS`, then let Cell 8's asserts pass.

## What to verify before trusting the numbers

- **The grain discriminator.** `group_mask()` guesses at a level/depth column, then at a
  null security id. Confirm which applies. Wrong here means every total double-counts.
- **The sector projection.** Cell 7 reports `direct:` or `ffill:`. `ffill` relies on STACH
  row order being hierarchical — cross-check the holdings-per-sector counts against the
  workstation for one composite. A mis-fill mislabels every holding without erroring.
- **Active weight exists.** `GROUPSALL` sets the row grain, not the column set. If the
  component only exposes portfolio and benchmark weight, either edit it or override
  `PACalculationParameters.columns`.
- **`snapshot` is True on the weights component** (printed by Cell 4c). If it is False,
  the component is a subperiod calculation and `Single` frequency at `0CQ` may not mean
  holdings *as of* the quarter end.
- **The grouping really is GICS sector**, not some other saved scheme — `get_pa_groups()`
  in Cell 4d. The projection is blind to what the grouping represents.
- **Cash rows.** Cash usually appears as a holding and may or may not sit inside a sector
  group. Check whether it lands with a sector, its own group, or none, and whether
  `cash_flag` distinguishes it. It affects any weight that should sum to 100%.
- **Ultimate parent** was "possibly" — it is looked for but not required. If it is wanted,
  confirm the component exposes it rather than assuming a null means no parent.

## Sources

Verified against **upstream `FactSet/enterprise-sdk` `main`** (PAEngine v3, SDK 4.0.0) —
`PACalculationParameters.md` (`accounts`/`benchmarks` as lists, `componentdetail` values
`GROUPS`/`GROUPSALL`/`TOTALS`/`SECURITIES`, `columns`, `groups`), `PAComponent.md`
(saved `accounts`, `benchmarks`, `dates`, `snapshot`), `PADateParameters.md`,
`PAIdentifier.md` (`holdingsmode` values), `PACalculationsApi.md`, `ComponentsApi.md`,
`DatesApi.md`, `ColumnsApi.md`, `GroupsApi.md`, `FrequenciesApi.md`, and `BREAKING.md`
(2026-07-21 `PADateParameters`; 2026-05-20 Python-wide bump).

Not against `code/python/PAEngine/v3/` in this repo, which is pinned at 2.2.2 and predates
the 4.0.0 changes.